In [19]:
# =============================================================
# 1 - Imports and environment setup
#
# What this does:
#   - loads all required libraries
#   - sets espeak-ng paths so phonemizer finds it on Windows
#   - verifies espeak works before running anything else
#
# Libraries used:
#   - torch / transformers : Wav2Vec2 neural network
#   - librosa              : audio loading and resampling
#   - phonemizer           : Python wrapper around espeak-ng
#   - Bio.Align            : Needleman-Wunsch sequence alignment
#   - sklearn              : precision, recall, F1 metrics
#   - pandas / numpy       : data manipulation
# =============================================================

import os
import json
import subprocess
import unicodedata
import warnings

import numpy as np
import pandas as pd
import torch
import librosa

from transformers import (
    AutoModelForCTC,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)
from huggingface_hub import snapshot_download
from phonemizer import phonemize
from Bio import Align
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# ── espeak-ng paths (Windows) ─────────────────────────────────
# phonemizer needs to know exactly where espeak-ng is installed
# because Windows does not always expose it automatically
ESPEAK_EXE = r"C:\Program Files\eSpeak NG\espeak-ng.exe"
ESPEAK_DLL = r"C:\Program Files\eSpeak NG\libespeak-ng.dll"

os.environ["PHONEMIZER_ESPEAK_PATH"]    = ESPEAK_EXE
os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = ESPEAK_DLL

# ── Paths ─────────────────────────────────────────────────────
MODEL_CACHE_DIR = r"C:\Users\Sarah\back2speak\model_cache"
AUDIO_DIR       = r"C:\Users\Sarah\Fichiers audio"
AUDIO_DB_PATH   = r"C:\Users\Sarah\audio_db.csv"
ITEMS_PATH      = r"C:\Users\Sarah\items.csv"

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# ── Quick sanity check ────────────────────────────────────────
print("Checking espeak-ng...")
result = subprocess.run(
    [ESPEAK_EXE, "--ipa", "-v", "fr", "chapeau"],
    capture_output=True, text=True, encoding="utf-8"
)
print(f"  espeak output for 'chapeau': {result.stdout.strip()}")
print(f"  expected: ʃapo")
print()
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
print("Environment ready.")

Checking espeak-ng...
  espeak output for 'chapeau': ʃapˈo
  expected: ʃapo

GPU available: True
GPU name:      NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU
Environment ready.


In [20]:
# =============================================================
# 2 - Load the French Wav2Vec2 phoneme model
#
# Model: Cnam-LMSSC/wav2vec2-french-phonemizer
#   - fine-tuned on Common Voice v13 French
#   - outputs IPA phonemes directly (not words)
#   - uses CTC decoding to collapse repeated phonemes
#
# Why we load it manually instead of from_pretrained():
#   The model's config files contain absolute paths from the
#   researcher's local machine (/home/zinc/...) which break
#   loading on any other machine. We fix those files before
#   loading.
#
# First run: downloads ~400MB to MODEL_CACHE_DIR
# Subsequent runs: loads from local cache instantly
# =============================================================

MODEL_ID = "Cnam-LMSSC/wav2vec2-french-phonemizer"

print(f"Downloading model files to: {MODEL_CACHE_DIR}")
print("First run takes ~2 minutes, then instant...")

snapshot_dir = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_CACHE_DIR,
    local_dir_use_symlinks=False   # required on Windows (no symlink privilege)
)
print(f"Model files at: {snapshot_dir}")
print()

# ── Fix broken config files ───────────────────────────────────
# Fix 1: tokenizer_config.json contains absolute paths
tok_path = os.path.join(snapshot_dir, "tokenizer_config.json")
with open(tok_path, "r", encoding="utf-8") as f:
    tok_cfg = json.load(f)
tok_cfg.pop("special_tokens_map_file", None)
tok_cfg.pop("name_or_path",            None)
tok_cfg.pop("tokenizer_file",          None)
tok_cfg["extra_special_tokens"] = {}
with open(tok_path, "w", encoding="utf-8") as f:
    json.dump(tok_cfg, f, ensure_ascii=False, indent=2)

# Fix 2: special_tokens_map.json stores tokens as dicts instead of strings
spm_path = os.path.join(snapshot_dir, "special_tokens_map.json")
with open(spm_path, "r", encoding="utf-8") as f:
    spm = json.load(f)
if "additional_special_tokens" in spm:
    spm["additional_special_tokens"] = [
        t["content"] if isinstance(t, dict) else t
        for t in spm["additional_special_tokens"]
    ]
with open(spm_path, "w", encoding="utf-8") as f:
    json.dump(spm, f, ensure_ascii=False, indent=2)

print("Config files fixed.")

# ── Load tokenizer ────────────────────────────────────────────
# We load directly from vocab.json to bypass all config files
# The tokenizer converts model output IDs back to IPA symbols
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=os.path.join(snapshot_dir, "vocab.json"),
    unk_token="[UNK]", pad_token="[PAD]",
    bos_token="<s>",   eos_token="</s>",
    word_delimiter_token="|", do_lower_case=False,
)

# ── Load feature extractor ────────────────────────────────────
# Normalizes the raw audio waveform before feeding to the model
# (zero mean, unit variance - required by Wav2Vec2)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000,
    padding_value=0.0, do_normalize=True,
    return_attention_mask=False,
)

# ── Build processor ───────────────────────────────────────────
# Combines feature_extractor + tokenizer into one object
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

# ── Load neural network weights ───────────────────────────────
# 94M parameter Wav2Vec2 model with CTC head
# Moved to GPU for ~10x faster inference
model  = AutoModelForCTC.from_pretrained(snapshot_dir)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

print()
print("=" * 50)
print(f"Model loaded on: {device}")
print(f"Parameters:      {sum(p.numel() for p in model.parameters()):,}")
vocab = tokenizer.get_vocab()
print(f"Vocabulary size: {len(vocab)} IPA phoneme tokens")
print()
print("Key French phonemes confirmed in vocabulary:")
for ph in ['ʃ', 's', 'k', 'ʒ', 'a', 'o', 'i', 'ʁ', 'ɛ']:
    print(f"  [{ph}] -> token ID {vocab.get(ph, 'MISSING')}")
print("=" * 50)

First run takes ~2 minutes, then instant...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

Model files at: C:\Users\Sarah\back2speak\model_cache

Config files fixed.

Model loaded on: cuda
Parameters:      94,418,621
Vocabulary size: 61 IPA phoneme tokens

Key French phonemes confirmed in vocabulary:
  [ʃ] -> token ID 46
  [s] -> token ID 17
  [k] -> token ID 10
  [ʒ] -> token ID 50
  [a] -> token ID 2
  [o] -> token ID 14
  [i] -> token ID 8
  [ʁ] -> token ID 45
  [ɛ] -> token ID 35


In [21]:
# =============================================================
# 3 - Load and merge the two CSV files
#
# CSV 1 — audio_db.csv (407 rows):
#   audio_id, speaker, age, sexe, position, type_item, decision
#   'decision' is the ground truth label:
#     'correct' / 'substitution_ch_vers_s' / 'distorsion' / 'indeterminé'
#
# CSV 2 — items.csv (31 rows):
#   item_id, mot/stimulus, type, position_du_/ʃ/,
#   Nombre de phonèmes, voyelle_précédente, voyelle_suivante
#
# Merge key:
#   audio filename 'P02_S02_W_F01.wav' -> item_id 'W_F01'
#   We extract the item_id from the filename and join on it
#
# Result: one merged dataframe with all metadata per audio file
# =============================================================

df_audio = pd.read_csv(r"C:\Users\Sarah\audio_db.csv")
df_items = pd.read_csv(r"C:\Users\Sarah\items.csv")

df_audio["decision"] = df_audio["decision"].str.strip()

# Clean item IDs in items CSV — remove leading bullets and dashes
df_items["item_id_clean"] = (
    df_items["item_id"]
    .astype(str).str.strip()
    .str.lstrip("•–- ").str.rstrip("–- ").str.strip()
)

def extract_item_id(audio_id):
    """
    Extract item_id from audio filename.
    'P02_S02_W_F01.wav' -> 'W_F01'
    'P02_S02_SYL01.wav' -> 'SYL01'
    'P02_S02_P04.wav'   -> 'P04'
    """
    parts = str(audio_id).replace(".wav", "").strip().split("_")
    return "_".join(parts[2:]) if len(parts) >= 3 else None

df_audio["item_id_extracted"] = df_audio["audio_id"].apply(extract_item_id)

# Merge audio metadata with item metadata
df = df_audio.merge(
    df_items,
    left_on="item_id_extracted",
    right_on="item_id_clean",
    how="left"
)

# Add full audio path and existence flag
df["audio_path"]  = df["audio_id"].apply(
    lambda x: os.path.join(AUDIO_DIR, x)
)
df["file_exists"] = df["audio_path"].apply(os.path.exists)

print(f"Audio DB rows:           {len(df_audio)}")
print(f"Items CSV rows:          {len(df_items)}")
print(f"Merged rows:             {len(df)}")
print(f"Files found on disk:     {df['file_exists'].sum()}")
print(f"Files missing on disk:   {(~df['file_exists']).sum()}")
print(f"Rows with target word:   {df['mot / stimulus'].notna().sum()}")
print()
print("Label distribution:")
print(df["decision"].value_counts().to_string())

Audio DB rows:           407
Items CSV rows:          31
Merged rows:             407
Files found on disk:     367
Files missing on disk:   40
Rows with target word:   386

Label distribution:
decision
correct                   384
substitution_ch_vers_s     17
distorsion                  5
indeterminé                 1


In [ ]:
# =============================================================
# 4 - Core helper functions
#
# These four functions form the backbone of the pipeline.
# They are defined once here and reused everywhere.
#
# 1. get_reference_phonemes(text)
#    Converts French text to IPA using espeak-ng.
#    This is what a perfect pronunciation should produce.
#    Uses MANUAL_REFERENCES for words espeak cannot handle
#    correctly (tee-shirt, ISO01).
#
# 2. split_ipa(ipa_string)
#    Splits an IPA string into a list of individual phoneme
#    characters. Handles combining diacritics correctly —
#    'ɔ̃' is one phoneme (ɔ + combining tilde) not two.
#
# 3. align_ipa(produced_str, reference_str)
#    Aligns two IPA strings at the character level using
#    Needleman-Wunsch. Returns the list of errors with type
#    (substitution / omission / insertion), position, and
#    the phonemes involved.
#
# 4. get_produced_phonemes(audio_path)
#    Runs Wav2Vec2 on an audio file and returns IPA string.
# =============================================================

# Words espeak-ng cannot phonemize correctly on its own
MANUAL_REFERENCES = {
    "CH prolongé (2 secondes)":   "ʃ",
    "– CH prolongé (2 secondes)": "ʃ",
    "tee-shirt":                  "tiʃɛʁt",
    "Il lave le tee-shirt":       "il lav lə tiʃɛʁt",
}


def get_reference_phonemes(text):
    """
    Convert a French word or phrase to IPA phonemes.

    Uses espeak-ng as the backend — a rule-based phonemizer
    that applies French pronunciation rules deterministically.
    Falls back to MANUAL_REFERENCES for known problem words.

    Input:  'chapeau'
    Output: 'ʃapo'
    """
    text_clean = str(text).strip().lstrip("–•- ").strip()
    if text_clean in MANUAL_REFERENCES:
        return MANUAL_REFERENCES[text_clean]
    if not text_clean:
        return ""
    try:
        return phonemize(
            text_clean,
            backend="espeak",
            language="fr-fr",
            with_stress=False,
            strip=True,
            preserve_punctuation=False
        ).strip()
    except Exception as e:
        print(f"  espeak error on '{text_clean}': {e}")
        return ""


def split_ipa(ipa_string):
    """
    Split an IPA string into a list of individual phoneme characters.

    This is not a simple string split because some IPA phonemes
    are represented as a base character + combining diacritic:
      'ɔ̃' = 'ɔ' (U+0254) + '̃' (U+0303 combining tilde)
    Splitting naively would break these into two tokens.

    Strategy: iterate character by character, attaching any
    Unicode 'Mark' category characters to the preceding base.

    Input:  'la fuʁʃɛt'
    Output: ['l', 'a', 'f', 'u', 'ʁ', 'ʃ', 'ɛ', 't']
    """
    ipa    = ipa_string.replace(" ", "")
    tokens = []
    i      = 0
    while i < len(ipa):
        combined = ipa[i]
        j        = i + 1
        while j < len(ipa) and unicodedata.category(ipa[j]).startswith("M"):
            combined += ipa[j]
            j        += 1
        tokens.append(combined)
        i = j
    return tokens


def align_ipa(produced_str, reference_str):
    """
    Align two IPA strings at the character level.

    Uses Needleman-Wunsch global alignment (the same algorithm
    used in bioinformatics to align DNA sequences). This finds
    the optimal mapping between produced and reference phonemes,
    including gaps for omissions and insertions.

    Scoring:
      match    = +2  (same phoneme)
      mismatch = -1  (different phoneme — substitution)
      gap open = -2  (missing or extra phoneme)
      gap ext  = -0.5

    Returns:
      errors       — list of dicts, one per error found
      aligned_prod — list of phonemes with '-' for gaps
      aligned_ref  — list of phonemes with '-' for gaps

    Error dict fields:
      type     : 'substitution' | 'omission' | 'insertion'
      position : index in the aligned sequence
      expected : reference phoneme (None for insertions)
      produced : produced phoneme (None for omissions)
    """
    prod_list = split_ipa(produced_str)
    ref_list  = split_ipa(reference_str)

    if not prod_list or not ref_list:
        return [], [], []

    aligner                  = Align.PairwiseAligner()
    aligner.mode             = "global"
    aligner.match_score      = 2
    aligner.mismatch_score   = -1
    aligner.open_gap_score   = -2
    aligner.extend_gap_score = -0.5

    best = next(iter(aligner.align(prod_list, ref_list)))

    aligned_prod, aligned_ref = [], []
    p_pos, r_pos = 0, 0

    for (p_start, p_end), (r_start, r_end) in zip(
        best.aligned[0], best.aligned[1]
    ):
        while p_pos < p_start:
            aligned_prod.append(prod_list[p_pos]); aligned_ref.append("-"); p_pos += 1
        while r_pos < r_start:
            aligned_prod.append("-"); aligned_ref.append(ref_list[r_pos]); r_pos += 1
        for pp, rr in zip(range(p_start, p_end), range(r_start, r_end)):
            aligned_prod.append(prod_list[pp])
            aligned_ref.append(ref_list[rr])
            p_pos += 1; r_pos += 1

    while p_pos < len(prod_list):
        aligned_prod.append(prod_list[p_pos]); aligned_ref.append("-"); p_pos += 1
    while r_pos < len(ref_list):
        aligned_prod.append("-"); aligned_ref.append(ref_list[r_pos]); r_pos += 1

    errors = []
    for i, (p, r) in enumerate(zip(aligned_prod, aligned_ref)):
        if p == r:
            continue
        elif p == "-":
            errors.append({"type": "omission",     "position": i,
                           "expected": r,    "produced": None})
        elif r == "-":
            errors.append({"type": "insertion",    "position": i,
                           "expected": None, "produced": p})
        else:
            errors.append({"type": "substitution", "position": i,
                           "expected": r,    "produced": p})

    return errors, aligned_prod, aligned_ref


def get_produced_phonemes(audio_path):
    """
    Run a WAV file through Wav2Vec2 and return IPA phonemes.

    Process:
      1. Load audio at 16kHz (Wav2Vec2 requires exactly 16kHz)
      2. Normalize with the processor (zero mean, unit variance)
      3. Run through the neural network -> logits (T x vocab_size)
      4. Argmax picks the best phoneme at each 20ms frame
      5. CTC decoding collapses repeated phonemes, removes blanks

    Input:  path to a WAV file
    Output: IPA string e.g. 'flɛs'
            empty string on error
    """
    try:
        waveform, _ = librosa.load(audio_path, sr=16000, mono=True)
        inputs      = processor(
            waveform, sampling_rate=16000,
            return_tensors="pt", padding=True
        )
        with torch.no_grad():
            logits = model(inputs.input_values.to(device)).logits
        ids = torch.argmax(logits, dim=-1)
        return processor.batch_decode(ids)[0].strip()
    except Exception as e:
        print(f"  Wav2Vec2 error on {os.path.basename(audio_path)}: {e}")
        return ""


print("All helper functions defined.")
print()

# Quick end-to-end test on one word
test_ref  = get_reference_phonemes("chapeau")
test_prod = "sɛ̃ po"   # simulate what a child with ch->s error produces
errors, ap, ar = align_ipa(test_prod, test_ref)
sh_errors = [e for e in errors if e["expected"] == "ʃ"]

print(f"Test — 'chapeau':")
print(f"  Reference IPA:    {test_ref}")
print(f"  Simulated output: {test_prod}")
print(f"  Aligned ref:      {ar}")
print(f"  Aligned prod:     {ap}")
print(f"  ʃ errors found:   {sh_errors}")

All helper functions defined.

Test — 'chapeau':
  Reference IPA:    ʃapo
  Simulated output: sɛ̃ po
  Aligned ref:      ['ʃ', 'a', 'p', 'o']
  Aligned prod:     ['s', 'ɛ̃', 'p', 'o']
  ʃ errors found:   [{'type': 'substitution', 'position': 0, 'expected': 'ʃ', 'produced': 's'}]


In [ ]:
# =============================================================
# 5 - analyze_audio(): full analysis of one audio file
#
# This is the main function. Given one WAV file and the French
# word the speaker was trying to say, it returns a complete
# structured report of every ʃ error found.
#
# For each ʃ error the function reports:
#   - which word it occurred in
#   - position in word: initial / medial / final
#   - the phoneme that was produced instead of ʃ
#   - the phoneme that precedes ʃ in the reference
#   - the phoneme that follows ʃ in the reference
#   - the error type: substitution / omission
#
# It also computes:
#   - PER (Phoneme Error Rate): edit distance / reference length
#   - success rate: 1 - (n_sh_errors / n_sh_in_reference)
#
# These outputs feed the ontology in the next step.
# =============================================================

def compute_per(produced, reference):
    """
    Compute Phoneme Error Rate between produced and reference IPA.
    PER = edit_distance / len(reference_phonemes)
    0.0 = perfect, 1.0 = completely wrong, >1.0 possible with insertions.
    """
    prod = produced.replace(" ", "")
    ref  = reference.replace(" ", "")
    if not ref:
        return 0.0
    m, n = len(prod), len(ref)
    dp   = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = (dp[i-1][j-1] if prod[i-1] == ref[j-1]
                        else 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]))
    return dp[m][n] / n

# Define French IPA vowels
VOWELS = {'a', 'e', 'i', 'o', 'u', 'y', 'ɛ', 'ɔ', 'œ', 'ø', 'ə', 
          'ɑ', 'ɐ', 'ɒ', 'ɪ', 'ʊ', 'ɛ̃', 'ɔ̃', 'ɑ̃', 'œ̃'}

def get_preceding_vowel(ref_chars, sh_idx):
    """
    Find the closest vowel that precedes ʃ in the reference.
    Skips consonants and looks backwards until it finds a vowel.
    Returns '(none)' if no vowel found before ʃ.
    """
    for i in range(sh_idx - 1, -1, -1):
        if ref_chars[i] in VOWELS:
            return ref_chars[i]
    return "(none)"

def get_following_vowel(ref_chars, sh_idx):
    """
    Find the closest vowel that follows ʃ in the reference.
    Skips consonants and looks forward until it finds a vowel.
    Returns '(none)' if no vowel found after ʃ.
    """
    for i in range(sh_idx + 1, len(ref_chars)):
        if ref_chars[i] in VOWELS:
            return ref_chars[i]
    return "(none)"

def analyze_audio(audio_path, target_word, label=None, verbose=True):
    """
    Full phoneme analysis of a single audio file.

    Parameters
    ----------
    audio_path  : str   — path to WAV file
    target_word : str   — French word/phrase the speaker intended
    label       : str   — ground truth label from CSV (optional)
    verbose     : bool  — whether to print the full report

    Returns
    -------
    dict with keys:
        audio_id        : filename
        target_word     : French text
        label           : ground truth label
        reference_ipa   : IPA from espeak (what should be said)
        produced_ipa    : IPA from Wav2Vec2 (what was detected)
        aligned_ref     : aligned reference character list
        aligned_prod    : aligned produced character list
        all_errors      : list of all phoneme errors
        sh_errors       : list of ʃ-specific errors with full context
        n_sh_in_ref     : number of ʃ phonemes expected in reference
        n_sh_errors     : number of ʃ errors detected
        success_rate    : 1 - (n_sh_errors / n_sh_in_ref)
        per             : phoneme error rate (0.0 to 1.0+)
        sh_detected     : True if at least one ʃ error was found
    """
    filename = os.path.basename(audio_path)

    # ── Step 1: get reference IPA from espeak ─────────────────
    reference_ipa = get_reference_phonemes(target_word)
    ref_chars     = split_ipa(reference_ipa)

    # ── Step 2: get produced IPA from Wav2Vec2 ────────────────
    produced_ipa = get_produced_phonemes(audio_path)

    # ── Step 3: align and detect all phoneme errors ───────────
    all_errors, aligned_prod, aligned_ref = align_ipa(
        produced_ipa, reference_ipa
    )

    # ── Step 4: filter to ʃ errors only ──────────────────────
    # For each ʃ error, compute full context:
    #   - position label in word (initial/medial/final)
    #   - preceding vowel in reference
    #   - following vowel in reference
    sh_indices = [i for i, c in enumerate(ref_chars) if c == "ʃ"]
    n_sh_in_ref = len(sh_indices)

    raw_sh_errors = [e for e in all_errors if e["expected"] == "ʃ"]

    sh_errors_detailed = []
    for idx, err in enumerate(raw_sh_errors):
        produced_ph = err["produced"] if err["produced"] else "(omitted)"

        # Find position of this ʃ in the reference character list
        sh_idx    = sh_indices[idx] if idx < len(sh_indices) else 0
        # preceding = ref_chars[sh_idx - 1] if sh_idx > 0                    else "(none)"
        # following = ref_chars[sh_idx + 1] if sh_idx < len(ref_chars) - 1   else "(none)"
        preceding = get_preceding_vowel(ref_chars, sh_idx)
        following = get_following_vowel(ref_chars, sh_idx)

        if sh_idx == 0:
            position_label = "initial"
        elif sh_idx == len(ref_chars) - 1:
            position_label = "final"
        else:
            position_label = "medial"

        sh_errors_detailed.append({
            "error_index":   idx + 1,
            "error_type":    err["type"],
            "expected":      "ʃ",
            "produced":      produced_ph,
            "position_label": position_label,
            "sh_index":      sh_idx,
            "preceding_vowel": preceding,   
            "following_vowel": following, 
            "target_word":   target_word,
        })

    # ── Step 5: compute metrics ───────────────────────────────
    per          = compute_per(produced_ipa, reference_ipa)
    n_sh_errors  = len(sh_errors_detailed)
    success_rate = (1 - n_sh_errors / n_sh_in_ref) if n_sh_in_ref > 0 else 1.0
    success_rate = max(0.0, success_rate)

    # ── Step 6: print report ──────────────────────────────────
    if verbose:
        sep = "=" * 65
        print(sep)
        print(f"FILE:           {filename}")
        print(f"TARGET WORD:    {target_word}")
        if label:
            print(f"GROUND TRUTH:   {label}")
        print(sep)
        print()
        print(f"Reference IPA   (what should be said):  {reference_ipa}")
        print(f"Produced IPA    (what Wav2Vec2 heard):  {produced_ipa}")
        print()
        print(f"Alignment:")
        print(f"  Reference: {aligned_ref}")
        print(f"  Produced:  {aligned_prod}")
        print()
        print(f"Phoneme Error Rate (PER): {per*100:.1f}%")
        print(f"ʃ phonemes expected:      {n_sh_in_ref}")
        print(f"ʃ errors detected:        {n_sh_errors}")
        print(f"Success rate:             {success_rate*100:.1f}%")
        print()

        if not sh_errors_detailed:
            print("  No ʃ errors detected — pronunciation correct.")
        else:
            for err in sh_errors_detailed:
                print(f"  Error {err['error_index']}:")
                print(f"    Type:             {err['error_type']}")
                print(f"    Expected:         [ʃ]")
                print(f"    Produced:         [{err['produced']}]")
                print(f"    Position in word: {err['position_label']}")
                print(f"    Preceded by vowel:   [{preceding}]")
                print(f"    Followed by vowel:   [{following}]")
                print()

        print(sep)

    return {
        "audio_id":       filename,
        "target_word":    target_word,
        "label":          label,
        "reference_ipa":  reference_ipa,
        "produced_ipa":   produced_ipa,
        "aligned_ref":    aligned_ref,
        "aligned_prod":   aligned_prod,
        "all_errors":     all_errors,
        "sh_errors":      sh_errors_detailed,
        "n_sh_in_ref":    n_sh_in_ref,
        "n_sh_errors":    n_sh_errors,
        "success_rate":   success_rate,
        "per":            per,
        "sh_detected":    n_sh_errors > 0,
    }


print("analyze_audio() defined.")

analyze_audio() defined.


In [34]:
# =============================================================
# 6 - Run the full pipeline on all 367 files
#
# This cell:
#   1. Runs analyze_audio() on every file found on disk
#   2. Collects all results into df_results
#   3. Computes global evaluation metrics:
#      - PER by speaker and label
#      - ʃ detection precision, recall, F1
#      - confusion matrix
#      - substitution analysis (what replaces ʃ)
#      - success rate distribution
# =============================================================

print("Running pipeline on all files...")
print("(This takes 5-10 minutes depending on GPU speed)")
print()

results = []

for i, row in df.iterrows():
    if not row["file_exists"]:
        continue

    target_word = str(row.get("mot / stimulus", "")).strip().lstrip("–•- ").strip()
    label       = row["decision"]

    result = analyze_audio(
        audio_path  = row["audio_path"],
        target_word = target_word,
        label       = label,
        verbose     = False    # suppress per-file printing for speed
    )

    result["speaker"]         = row["speaker"]
    result["age"]             = row["age"]
    result["position_du_sh"]  = str(row.get("position_du_/ʃ/", ""))
    result["voyelle_suivante"] = str(row.get("voyelle_suivante", ""))
    result["is_correct"]      = (label == "correct")

    results.append(result)

    if len(results) % 50 == 0:
        print(f"  {len(results)} files processed...")

df_results = pd.DataFrame(results)

print()
print(f"Done. {len(df_results)} files processed.")
print()

# ── Global metrics ────────────────────────────────────────────

print("=" * 65)
print("METRIC 1 — Phoneme Error Rate (PER)")
print("  Measures how different the produced phonemes are from")
print("  the reference. 0% = perfect, 100% = completely wrong.")
print("=" * 65)
df_no_iso = df_results[df_results["position_du_sh"] != "Isolée"]
print(f"  All files (excl. ISO01):    {df_no_iso['per'].mean()*100:.1f}%")
print(f"  Correct pronunciations:     {df_no_iso[df_no_iso['is_correct']]['per'].mean()*100:.1f}%")
print(f"  Incorrect pronunciations:   {df_no_iso[~df_no_iso['is_correct']]['per'].mean()*100:.1f}%")
print()

print("=" * 65)
print("METRIC 2 — ʃ Detection (Precision / Recall / F1)")
print("  Measures how well the pipeline identifies mispronounced ʃ.")
print("  y_true = 1 if label is 'correct' (ʃ should be present)")
print("  y_pred = 1 if model detected ʃ in produced phonemes")
print("=" * 65)
y_true = df_results["is_correct"].astype(int).values
y_pred = df_results["has_sh_produced"].astype(int).values \
         if "has_sh_produced" in df_results.columns \
         else df_results["produced_ipa"].str.contains("ʃ").astype(int).values

print(classification_report(
    y_true, y_pred,
    target_names=["incorrect pronunciation", "correct pronunciation"]
))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print(f"                        predicted no-ʃ   predicted ʃ")
print(f"  actual incorrect      {cm[0][0]:<16}   {cm[0][1]}")
print(f"  actual correct        {cm[1][0]:<16}   {cm[1][1]}")
print()

print("=" * 65)
print("METRIC 3 — PER by speaker")
print("=" * 65)
speaker_stats = df_results.groupby("speaker").agg(
    n_files       = ("audio_id",    "count"),
    mean_per      = ("per",         lambda x: f"{x.mean()*100:.1f}%"),
    n_incorrect   = ("is_correct",  lambda x: (~x).sum()),
    mean_success  = ("success_rate",lambda x: f"{x.mean()*100:.1f}%"),
)
print(speaker_stats.to_string())
print()

print("=" * 65)
print("METRIC 4 — Substitution analysis on incorrect files")
print("  What phoneme was produced instead of ʃ?")
print("=" * 65)

all_sh_errors = []
for _, row in df_results[~df_results["is_correct"]].iterrows():
    for err in row["sh_errors"]:
        all_sh_errors.append(err)

if all_sh_errors:
    df_errors = pd.DataFrame(all_sh_errors)
    print("By substituted phoneme:")
    print(df_errors.groupby("produced")["target_word"].count()
          .sort_values(ascending=False).to_string())
    print()
    print("By position in word:")
    print(df_errors.groupby("position_label")["target_word"].count()
          .sort_values(ascending=False).to_string())
    print()
    print("By following vowel:")
    print(df_errors.groupby("following_vowel")["target_word"].count()
          .sort_values(ascending=False).to_string())
    print()
    print("By preceding vowel:")
    print(df_errors.groupby("preceding_vowel")["target_word"].count()
          .sort_values(ascending=False).to_string())

Running pipeline on all files...
(This takes 5-10 minutes depending on GPU speed)

  50 files processed...
  100 files processed...
  150 files processed...
  200 files processed...
  250 files processed...
  300 files processed...
  350 files processed...

Done. 367 files processed.

METRIC 1 — Phoneme Error Rate (PER)
  Measures how different the produced phonemes are from
  the reference. 0% = perfect, 100% = completely wrong.
  All files (excl. ISO01):    14.8%
  Correct pronunciations:     11.8%
  Incorrect pronunciations:   60.7%

METRIC 2 — ʃ Detection (Precision / Recall / F1)
  Measures how well the pipeline identifies mispronounced ʃ.
  y_true = 1 if label is 'correct' (ʃ should be present)
  y_pred = 1 if model detected ʃ in produced phonemes
                         precision    recall  f1-score   support

incorrect pronunciation       0.49      0.96      0.65        23
  correct pronunciation       1.00      0.93      0.96       344

               accuracy                

In [35]:
# =============================================================
# 7 - Test analyze_audio() on a single file
#
# Use this to inspect one file in detail.
# Change the path and target_word to test any file.
# This also works for new recordings with no metadata —
# just provide the path and what word the speaker tried to say.
# =============================================================

# ── Test on an incorrect file ─────────────────────────────────
result = analyze_audio(
    audio_path  = r"C:\Users\Sarah\Fichier audio\P03_S01_W_F02.wav",
    target_word = "fleche",
    label       = "substitution_ch_vers_s",
    verbose     = True
)

print()
print("Structured output (for ontology input):")
print(f"  n_sh_errors:   {result['n_sh_errors']}")
print(f"  success_rate:  {result['success_rate']*100:.1f}%")
print(f"  per:           {result['per']*100:.1f}%")
for err in result["sh_errors"]:
    print(f"  error:         ʃ -> [{err['produced']}]"
          f"  position={err['position_label']}"
          f"  preceded_by=[{err['preceded_by']}]"
          f"  followed_by=[{err['followed_by']}]")

  Wav2Vec2 error on P03_S01_W_F02.wav: [Errno 2] No such file or directory: 'C:\\Users\\Sarah\\Fichier audio\\P03_S01_W_F02.wav'
FILE:           P03_S01_W_F02.wav
TARGET WORD:    fleche
GROUND TRUTH:   substitution_ch_vers_s

Reference IPA   (what should be said):  flɛʃ
Produced IPA    (what Wav2Vec2 heard):  

Alignment:
  Reference: []
  Produced:  []

Phoneme Error Rate (PER): 100.0%
ʃ phonemes expected:      1
ʃ errors detected:        0
Success rate:             100.0%

  No ʃ errors detected — pronunciation correct.

Structured output (for ontology input):
  n_sh_errors:   0
  success_rate:  100.0%
  per:           100.0%
